# Colab preparation

This cell gets the latest code from GitHub (clone on first run, `git pull` on later runs), mounts Google Drive, and installs only missing packages. It does not train or draw the figure.

Run this cell again after we push updates so Colab picks up new `.py` files. Accept the Drive prompt when it appears. The Task 1.1 data is at `My Drive/Foundation Model/vendor/pfdelta_task1.1`.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    from google.colab import drive

    drive.mount("/content/drive")

missing = []
for module, package in (
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("pyarrow", "pyarrow"),
    ("huggingface_hub", "huggingface_hub"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Required packages are already installed.")


# Training

This cell trains **TPGT** only (type-pair graph Transformer, independent weights per stage) on that Drive dataset. It writes checkpoints to `Foundation Model/tpgt_task11` and comparison tables to `Foundation Model/results`. It does not draw the figure.

Use a **GPU** runtime. A finished seed is skipped if this cell is run again. The TPGT sizes are implementation choices for Colab, not the authors' hyperparameters: hidden size 32, 2 heads, 3 unshared layers, batch 16, Adam learning rate 1e-4, 5 epochs, seeds 0, 1, and 2. IEEE 118 pair features use a slack-rooted spanning tree because the draft's path impedance assumes a radial feeder.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
if not (here / "train_tpgt_task11.py").is_file():
    if on_colab():
        if not REPO.is_dir():
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
        os.chdir(REPO)
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

%run train_tpgt_task11.py


# SVG figure

This cell draws the published Task 1.1 SVG (GENCO Base, CANOS-PF, GNS-S, PFNet) from the published table. It does not train.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
if not (here / "plot_task1_1.py").is_file():
    if on_colab():
        if not REPO.is_dir():
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
        os.chdir(REPO)
    elif (here / "genco" / "plot_task1_1.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("plot_task1_1.py is not in this folder.")

%run plot_task1_1.py --published
